# 06. Xe II: расширенная модель (+ 6d, 7p, 4f) — каскады и устойчивость σ

Модель `xe2_ext` (`models.py`): мишень Ванга + 5p⁴6d, 7p, 4f, корреляционная 7d, все уровни
до 19.2 эВ (до (¹D₂)6d). Уровни — NIST ASD (таблица Ванга кончается на 18.6 эВ); номера
переводятся в номера Ванга по энергии (`nb.level_number`, допуск 1 см⁻¹), у новых уровней
номера Ванга нет.

1. **Энергии и A** новых уровней против NIST.
2. **Устойчивость σ**: σ(1→u), σ(2→u) для выбранных u — расширенная модель против модели
   Ванга (ноутбук 05) и Ванга 2019. Разница — оценка неопределённости σ от выбора мишени.
3. **Каскады** с учётом 6d (и цепочек 7p → 7s/6d → 6p, 4f → 5d → 6p): P(h→u), k_casc, отношение
   к прямому k — как в 05; отдельно вклад уровней, которых нет в модели Ванга.

Все σ — DBSR (этот расчёт); ветвления — DBSR E1 (этот расчёт).

**Время**: состояний больше, чем в 05, — расчёт дольше (сервер). Сначала можно запустить
`stage="transitions"` (минуты): пункты 1 и P(h→u) из пункта 3 считаются без рассеяния.

**Выход** (папка `xe2_ext`): `sigma_xe2_ext.xlsx`, `rates_xe2_ext.xlsx` (номера NIST ASD),
`ext_vs_wang.csv`, `xe2_ext_cascade_P.csv`, `xe2_ext_k_cascade.xlsx` (+ .csv).

In [ ]:
# ---------- параметры (поменяйте пути под себя)
from pathlib import Path
import sys
REPO    = Path("/data/pydbsr_calc/DBSR_src")          # клон репозитория pydbsr
CRM     = REPO / "examples" / "crm"                   # nbtools.py, run_model.py, models.py
NIST    = REPO / "docs" / "crm"                       # выгрузки NIST ASD (nist_*.csv)
DATA    = Path("/data/pydbsr_calc/data")              # ваши файлы
XLSX    = DATA / "CrossSectionsIon_3.xlsx"            # Wang et al (2019), сечения e + Xe+
RUNS    = Path("/data/pydbsr_calc/runs/crm")          # результаты расчётов (большой диск)
SCRATCH, SCRATCH_GB = Path.home() / "pydbsr_tmp", 5   # временные файлы dbsr_mat (быстрый диск)
CORES, MEM, HD_THREADS = 32, 100, 8                   # ядра, ГБ памяти, потоки диагонализации
TE = [0.3, 0.5, 0.7, 1.0, 1.5, 2.0, 3.0, 5.0, 7.0, 10.0, 15.0, 20.0]   # сетка Te (эВ) для констант скоростей
sys.path.insert(0, str(CRM))
import numpy as np, matplotlib.pyplot as plt
import pydbsr as db
from pydbsr import rates, reference
import nbtools as nb
%matplotlib inline

In [ ]:
JMAX, EMAX = 25, 60.0
STAGE = "all"                 # "transitions" — только структура (быстро)
W = RUNS / "xe2_ext"
job = nb.Job(W, "xe2_ext", stage=STAGE, jmax=JMAX, cores=CORES, mem=MEM, hd_threads=HD_THREADS,
             scratch=SCRATCH, scratch_gb=SCRATCH_GB, emax=EMAX, de=0.0136, multipoles="E1",
             sigma_initial="1,2", te=",".join(map(str, TE)))
job.start()

In [ ]:
job.status()          # ход расчёта (запускайте когда хотите)

## 1. Энергии и номера Ванга

In [ ]:
print((W / "target_table.txt").read_text())
ref = reference.read_xlsx(XLSX)
elv = [lv.energy_cm for lv in ref.levels]
tg = db.Target.load(W / "target")
S = {s.nist_no: s for s in tg.states if s.nist_no}           # номер NIST ASD -> состояние
wang = {n: nb.level_number(elv, s.exp_energy_cm, tol=1.0) for n, s in S.items()}
Ecm = {n: s.exp_energy_cm for n, s in S.items()}
lab = {n: s.nist_label or s.config for n, s in S.items()}
of_wang = {w: n for n, w in wang.items() if w}
new = [n for n in sorted(S) if not wang[n]]
print(f"{len(S)} состояний; без номера Ванга (новые): {len(new)}")
for n in new:
    print(f"   ASD {n:3d}  {lab[n]:34s} {Ecm[n] / nb.CM_PER_EV:7.3f} эВ")

## 2. A новых уровней против NIST

In [ ]:
nistA = [x for f in ("nist_XeII.csv", "nist_XeII_UV.csv", "nist_XeII_IR.csv") if (NIST / f).exists()
         for x in nb.nist_lines(NIST / f) if x["A"]]
q = []
for r in nb.read_csv(W / "transitions_E1.csv"):
    if not r["upper_no"] or not r["lower_no"]:
        continue
    u, l = int(r["upper_no"]), int(r["lower_no"])
    hit = [x for x in nistA if x["Ek"] and x["Ei"] and abs(x["Ek"] - Ecm[u]) < 1 and abs(x["Ei"] - Ecm[l]) < 1]
    if hit:
        a = float(r["A_exp"]); q.append(a / hit[0]["A"])
        if u in new:
            print(f"{hit[0]['wl']:9.3f} {lab[u]:30s} -> {lab[l]:30s} DBSR {a:.2e}  NIST {hit[0]['A']:.2e}  {a / hit[0]['A']:.2f}")
q = np.array(q)
print(f"все линии с A в NIST: {len(q)}, медиана DBSR/NIST = {np.median(q):.2f}, в пределах ×2: {np.mean((q > 0.5) & (q < 2)):.0%}")

## 3. σ(1→u), σ(2→u): расширенная модель, модель Ванга (05), Ванг 2019

k_ext/k_Ванг и k_ext/k_05 по Te. Если 05 не досчитан, его столбцы пропускаются.

In [ ]:
import csv
cs = db.CollisionStrengths.load(W / f"omega_J{JMAX:g}.npz")
def ext_sigma(i, j):
    E, s = cs.incident_energy(S[i].name), cs.sigma(S[i].name, S[j].name)
    m = np.isfinite(s) & (E > 0)
    return E[m], s[m]
def thr(i, j):
    return (Ecm[j] - Ecm[i]) / nb.CM_PER_EV
f05 = RUNS / "xe2_wang_full" / f"omega_J{JMAX:g}.npz"
cs05 = db.CollisionStrengths.load(f05) if f05.exists() else None
if cs05:
    tg05 = db.Target.load(RUNS / "xe2_wang_full" / "target")
    n05 = {s.nist_no: s.name for s in tg05.states if s.nist_no and s.name in cs05.names}

TEa = np.array(TE)
def k_of(E, s, thr_ev):
    """Maxwellian <sigma v> (cm3/s) on the TE grid; tail above the last energy: 1/E."""
    return rates.rates_on_grid(E, s, TEa, thr_ev)
def show_k(title, k):
    print(f"{title:28s} " + "  ".join(f"{v:9.2e}" for v in k))
print(" " * 28 + "  ".join(f"Te={t:<6g}" for t in TEa))

rows = []
fig, axs = plt.subplots(2, len(nb.XE2_LINES), figsize=(4 * len(nb.XE2_LINES), 6.5), squeeze=False)
for col, x in enumerate(nb.XE2_LINES):
    u = of_wang[x["upper"]]
    for row, i in enumerate((1, 2)):
        ax = axs[row][col]
        E, s = ext_sigma(i, u); ke = k_of(E, s, thr(i, u)); ax.plot(E, s / 1e-16, "r", lw=0.6, label="xe2_ext")
        r = dict(i=i, upper_wang=x["upper"], line_nm=x["wl"])
        if (i, x["upper"]) in ref.sigma:
            Ew, sw = ref.sigma[(i, x["upper"])]; ax.plot(Ew, sw / 1e-16, "k", lw=0.8, label="Ванг 2019")
            r.update({f"ext/Wang_Te{t:g}": v for t, v in zip(TEa, ke / k_of(Ew, sw, thr(i, u)))})
        if cs05:
            E5, s5 = cs05.incident_energy(n05[i]), cs05.sigma(n05[i], n05[x["upper"]])
            m = np.isfinite(s5) & (E5 > 0); ax.plot(E5[m], s5[m] / 1e-16, "b", lw=0.6, label="05 (модель Ванга)")
            r.update({f"ext/05_Te{t:g}": v for t, v in zip(TEa, ke / k_of(E5[m], s5[m], thr(i, u)))})
        rows.append(r)
        ax.set_xscale("log"); ax.set_title(f"{i} → {x['upper']} ({x['wl']} нм)", fontsize=9)
axs[0][0].legend(fontsize=7); fig.supxlabel("E, эВ"); fig.supylabel("σ, 10⁻¹⁶ см²"); fig.tight_layout(); plt.show()
keys = sorted({k for r in rows for k in r}, key=lambda k: (k not in ("i", "upper_wang", "line_nm"), k))
with open(W / "ext_vs_wang.csv", "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=keys); w.writeheader(); w.writerows(rows)
for r in rows:
    print(r["i"], "->", r["upper_wang"], "  ".join(f"{k}={v:.2f}" for k, v in r.items() if "Te" in k))

## 4. Каскады с 6d, 7p, 4f

Как в 05, но по всем уровням расширенной модели. Колонка «новый» — уровня нет в модели
Ванга; `share_new` — доля каскада от таких уровней.

In [ ]:
P_MIN = 1e-3
br, tot, A = nb.branching_from_csv(W / "transitions_E1.csv")
P = nb.cascade_prob(br)
kcache = {}
def k_pair(i, j):
    if (i, j) not in kcache:
        kcache[(i, j)] = k_of(*ext_sigma(i, j), thr(i, j))
    return kcache[(i, j)]
prow, k_c, lab_c = [], {}, {}
for x in nb.XE2_LINES:
    u = of_wang[x["upper"]]
    feed = sorted(((h, P[h][u]) for h in P if P[h].get(u, 0) > P_MIN and h in S), key=lambda t: -t[1])
    print(f"\n{x['wl']} нм, уровень Ванга {x['upper']} (ASD {u}): питают {len(feed)} уровней")
    for h, p in feed:
        prow.append(dict(upper_wang=x["upper"], upper_asd=u, line_nm=x["wl"], h_asd=h, h_wang=wang[h] or "",
                         h_label=lab[h], new=h in new, P=p))
        print(f"   ASD {h:3d} {lab[h]:34s} P = {p:.3f}" + ("   новый" if h in new else ""))
    for i in (1, 2):
        kc = sum(k_pair(i, h) * p for h, p in feed)
        kn = sum(k_pair(i, h) * p for h, p in feed if h in new)
        kd = k_pair(i, u)
        for tag, v in (("casc", kc), ("ratio", kc / kd), ("share_new", kn / np.maximum(kc, 1e-300))):
            k_c[(i, x["upper"], tag)] = v
            lab_c[(i, x["upper"], tag)] = f"{i}->{x['upper']} " + {"casc": "cascade", "ratio": "cascade/direct",
                                                                   "share_new": "share of new levels"}[tag]
        show_k(f"  из {i}: k_casc/k_прям", kc / kd)
        show_k(f"  из {i}: доля новых", kn / np.maximum(kc, 1e-300))
with open(W / "xe2_ext_cascade_P.csv", "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(prow[0])); w.writeheader(); w.writerows(prow)
reference.write_rates_xlsx(W / "xe2_ext_k_cascade.xlsx", TEa, k_c, lab_c,
                           note="cascade into Wang levels u: sum_h <sigma v>(i->h) P(h->u), cm3/s; DBSR, model xe2_ext")
print("->", W / "xe2_ext_cascade_P.csv", W / "xe2_ext_k_cascade.xlsx")